# Anonymisation RGPD du dataset RH
**Hackathon IA × RH | Thème : Cybersécurité**

Ce notebook applique les techniques d'anonymisation RGPD sur le dataset RH brut (v14) avant tout traitement ML.

### Techniques appliquées
| Technique | Champs concernés |
|---|---|
| Pseudonymisation | `Employee_Name`, `EmpID` |
| Suppression | `Zip`, `State` |
| Généralisation | `DOB` → tranche d'âge, `Salary` → fourchette |
| Masquage partiel | `ManagerName` |


In [53]:
import pandas as pd
import numpy as np
import hashlib
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('HRDataset_v14.csv')
print(f"Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")
df.head(3)


Dataset chargé : 311 lignes, 36 colonnes


,Employee_Name,EmpID,MarriedID,MaritalStatusID,GenderID,EmpStatusID,DeptID,PerfScoreID,FromDiversityJobFairID,Salary,...,ManagerName,ManagerID,RecruitmentSource,PerformanceScore,EngagementSurvey,EmpSatisfaction,SpecialProjectsCount,LastPerformanceReview_Date,DaysLateLast30,Absences
0,"Adinolfi, Wilson K",10026,0,0,1,1,5,4,0,62506,...,Michael Albert,22.0,LinkedIn,Exceeds,4.60,5,0,1/17/2019,0,1
1,"Ait Sidi, Karthikeyan",10084,1,1,1,5,3,3,0,104437,...,Simon Roup,4.0,Indeed,Fully Meets,4.96,3,6,2/24/2016,0,17
2,"Akinkuolie, Sarah",10196,1,1,0,5,5,3,0,64955,...,Kissy Sullivan,20.0,LinkedIn,Fully Meets,3.02,3,0,5/15/2012,0,3


## 1. Identification des données sensibles

In [56]:
# We remove the sensitive features
sensitive_direct  = ['Employee_Name', 'EmpID', 'DOB', 'Zip', 'State']
sensitive_indirect = ['Sex', 'RaceDesc', 'HispanicLatino', 'MaritalDesc', 'CitizenDesc', 'Salary', 'ManagerName']

print(" Directly Sensitive data :")
print(df[sensitive_direct].head(5).to_string())
print("\n Indirectly Sensitive data :")
print(df[sensitive_indirect].head(5).to_string())


 Directly Sensitive data :
              Employee_Name  EmpID       DOB   Zip State
0       Adinolfi, Wilson  K  10026  07/10/83  1960    MA
1  Ait Sidi, Karthikeyan     10084  05/05/75  2148    MA
2         Akinkuolie, Sarah  10196  09/19/88  1810    MA
3              Alagbe,Trina  10088  09/27/88  1886    MA
4          Anderson, Carol   10069  09/08/89  2169    MA

 Indirectly Sensitive data :
  Sex RaceDesc HispanicLatino MaritalDesc CitizenDesc  Salary     ManagerName
0  M     White             No      Single  US Citizen   62506  Michael Albert
1  M     White             No     Married  US Citizen  104437      Simon Roup
2   F    White             No     Married  US Citizen   64955  Kissy Sullivan
3   F    White             No     Married  US Citizen   64991    Elijiah Gray
4   F    White             No    Divorced  US Citizen   50825  Webster Butler


## 2. Pseudonymisation des identifiants directs

In [59]:
def pseudonymize(value, salt="hackathon_rh_2024"):
    """We tried to anonymize sensitive but important data"""
    h = hashlib.sha256(f"{salt}{value}".encode()).hexdigest()
    return f"EMP_{h[:8].upper()}"

df_anon = df.copy()

# We used pseudonym for the names and IDs
df_anon['Employee_Name'] = df_anon['Employee_Name'].apply(pseudonymize)
df_anon['EmpID']         = df_anon['EmpID'].apply(lambda x: pseudonymize(str(x)))
df_anon['ManagerName']   = df_anon['ManagerName'].apply(lambda x: pseudonymize(str(x), salt="mgr"))

print("Example :")
print(df_anon[['Employee_Name','EmpID','ManagerName']].head(5))


Example :
  Employee_Name         EmpID   ManagerName
0  EMP_03F0ACCF  EMP_EF5A10A2  EMP_B85475B8
1  EMP_022A00D3  EMP_D9D3615D  EMP_A1A4858B
2  EMP_D32F7EC4  EMP_658AFDEA  EMP_7E5EF670
3  EMP_45F65357  EMP_8C644B03  EMP_935794E6
4  EMP_9C8241E6  EMP_0A1CDC43  EMP_DEC54272


## 3. Suppression des quasi-identifiants géographiques

In [62]:
# We removed the zip and state
df_anon.drop(columns=['Zip', 'State'], inplace=True)


## 4. Généralisation — DOB → tranche d'âge

In [65]:
def age_to_bracket(age):
    if pd.isna(age): return 'Inconnu'
    age = int(age)
    low = (age // 5) * 5
    return f"{low}-{low+4}"

# Calcul de l'âge en 2019 à partir de DOB
ref_date = pd.Timestamp('2019-01-01')
df_anon['DOB'] = pd.to_datetime(df_anon['DOB'], errors='coerce')
df_anon['age_2019'] = ((ref_date - df_anon['DOB']).dt.days / 365.25)
df_anon['age_bracket'] = df_anon['age_2019'].apply(age_to_bracket)

# Supprimer DOB et age brut
df_anon.drop(columns=['DOB', 'age_2019'], inplace=True)

print("Distribution des tranches d'âge :")
print(df_anon['age_bracket'].value_counts())


Distribution des tranches d'âge :
age_bracket
30-34      87
35-39      71
-55--51    42
40-44      28
-50--46    27
25-29      25
-60--56     9
-45--41     9
-35--31     8
-40--36     5
Name: count, dtype: int64


## 5. Généralisation — Salary → fourchette salariale

In [68]:
def salary_to_bracket(salary):
    if pd.isna(salary): return 'unk'
    low = (int(salary) // 10000) * 10000
    return f"{low//1000}k-{(low+10000)//1000}k"

df_anon['salary_bracket'] = df_anon['Salary'].apply(salary_to_bracket)
df_anon.drop(columns=['Salary'], inplace=True)

print("Distribution des fourchettes salariales :")
print(df_anon['salary_bracket'].value_counts())


Distribution des fourchettes salariales :
salary_bracket
60k-70k      101
50k-60k       90
40k-50k       31
70k-80k       31
80k-90k       17
90k-100k      16
100k-110k     10
110k-120k      4
170k-180k      2
150k-160k      2
140k-150k      2
130k-140k      1
180k-190k      1
250k-260k      1
120k-130k      1
220k-230k      1
Name: count, dtype: int64


## 6. Sauvegarde du dataset anonymisé

In [77]:
df_anon.to_csv('HRDataset_anonymized.csv', index=False)
print(f"Final dataset : {df_anon.shape[0]} lignes, {df_anon.shape[1]} colonnes")
print("\nFeatures :")
print(list(df_anon.columns))


Final dataset : 311 lignes, 34 colonnes

Features :
['Employee_Name', 'EmpID', 'MarriedID', 'MaritalStatusID', 'GenderID', 'EmpStatusID', 'DeptID', 'PerfScoreID', 'FromDiversityJobFairID', 'Termd', 'PositionID', 'Position', 'Sex', 'MaritalDesc', 'CitizenDesc', 'HispanicLatino', 'RaceDesc', 'DateofHire', 'DateofTermination', 'TermReason', 'EmploymentStatus', 'Department', 'ManagerName', 'ManagerID', 'RecruitmentSource', 'PerformanceScore', 'EngagementSurvey', 'EmpSatisfaction', 'SpecialProjectsCount', 'LastPerformanceReview_Date', 'DaysLateLast30', 'Absences', 'age_bracket', 'salary_bracket']
